# Nica-GeoFetch: unidades hidrográficas Pfafstetter de Nicaragua

Este notebook obtiene unidades hidrográficas oficiales de INETER. Usted puede descargar el KML original o preparar formatos para análisis geoespacial. **No necesita conocer Python para usar los controles principales.**

**Promesa del flujo:** Nica-GeoFetch descarga y conserva los KML oficiales originales. Después revisa su geometría y, cuando se solicita, prepara formatos analíticos como GeoPackage, GeoJSON y Shapefile. La reparación geométrica es opcional, explícita y queda registrada.

Siga los pasos numerados. El nivel 4 está seleccionado inicialmente; también puede obtener los niveles 4, 5, 6 y 7 en una sola acción.

> Los datos institucionales son material de terceros y no están cubiertos por la licencia Apache-2.0 del software. No se ha identificado una licencia explícita de datos abiertos. Consulte a INETER antes de redistribuir copias completas.

`/content` es almacenamiento temporal dentro de esta sesión de Colab: no se guarda automáticamente en su computadora y desaparece cuando termina la sesión. Descargue el ZIP final antes de cerrar, o seleccione Google Drive. Drive se monta únicamente después de una selección explícita.

## 1. Instalar Nica-GeoFetch

De forma predeterminada se instala desde `https://github.com/datanicaragua/nica-geofetch` usando la referencia configurable `GIT_REF`.

- Antes de la primera versión se usa `main`.
- Después de publicar versiones, cambie `GIT_REF` por una etiqueta estable, por ejemplo `v0.1.0`, para obtener resultados reproducibles.
- La instalación anónima desde GitHub requiere que el repositorio sea público.
- Para probar mientras el repositorio sea privado, cambie `INSTALL_SOURCE` a `"zip"`; Colab solicitará un ZIP del paquete o repositorio.
- No pegue tokens de GitHub ni otras credenciales en este cuaderno público.
- Si la instalación falla, la celda se detendrá con una explicación en español antes de importar el paquete.

In [ ]:
import subprocess
import sys

REPOSITORY_URL = "https://github.com/datanicaragua/nica-geofetch"
GIT_REF = "main"  # Antes de v0.1.0; luego use una etiqueta estable.
INSTALL_SOURCE = "github"  # Opciones: "github" o "zip".
BOOTSTRAP_OK = False


PRIVATE_REPOSITORY_GUIDANCE = (
    "La instalación anónima desde GitHub requiere que el repositorio sea público. "
    'Para probar un repositorio privado, cambie INSTALL_SOURCE a "zip" y cargue '
    "el paquete manualmente. No pegue tokens de GitHub ni credenciales en este cuaderno."
)


def run_pip_install(requirement):
    command = [sys.executable, "-m", "pip", "install", "-q", requirement]
    try:
        result = subprocess.run(
            command,
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(
            "No se pudo iniciar pip en este entorno. Verifique que Python y pip estén disponibles."
        ) from exc
    if result.returncode == 0:
        return

    diagnostic = f"{result.stderr}\n{result.stdout}".lower()
    if any(
        marker in diagnostic
        for marker in (
            "cannot find command 'git'",
            "git was not found",
            "'git' is not recognized",
            "no such file or directory: 'git'",
        )
    ):
        reason = (
            "Git no está disponible en este entorno, por lo que pip no puede instalar desde GitHub."
        )
    elif any(
        marker in diagnostic
        for marker in (
            "authentication failed",
            "permission denied",
            "could not read username",
            "terminal prompts disabled",
            "403",
        )
    ):
        reason = "GitHub rechazó la autenticación para el repositorio solicitado."
    elif any(
        marker in diagnostic
        for marker in (
            "repository not found",
            "404",
            "could not resolve host",
            "failed to connect",
        )
    ):
        reason = "El repositorio no está disponible de forma anónima, sigue privado o no puede alcanzarse."
    else:
        reason = f"pip no pudo instalar el paquete (código de salida {result.returncode})."
    raise RuntimeError(f"{reason}\n\n{PRIVATE_REPOSITORY_GUIDANCE}")


def bootstrap_package():
    if INSTALL_SOURCE == "github":
        requirement = f"nica-geofetch[notebook] @ git+{REPOSITORY_URL}.git@{GIT_REF}"
        run_pip_install(requirement)
        return "github"
    if INSTALL_SOURCE == "zip":
        from google.colab import files

        print("Seleccione un ZIP del paquete o del repositorio Nica-GeoFetch.")
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith((".zip", ".whl"))]
        if len(candidates) != 1:
            raise ValueError("Cargue exactamente un archivo .zip o .whl del paquete.")
        run_pip_install(candidates[0])
        run_pip_install("ipywidgets>=8.1,<9")
        return f"zip: {candidates[0]}"
    raise ValueError('INSTALL_SOURCE debe ser "github" o "zip".')


installation_source = bootstrap_package()
try:
    import nica_geofetch
except ModuleNotFoundError:
    raise RuntimeError(
        "La instalación finalizó, pero Nica-GeoFetch todavía no puede importarse. "
        "Revise la salida de pip, reinicie el entorno y vuelva a ejecutar esta celda."
    ) from None

BOOTSTRAP_OK = True
print(f"Versión instalada: {nica_geofetch.__version__}")
print(f"Referencia Git seleccionada: {GIT_REF}")
print(f"Fuente de instalación: {installation_source}")

## 2. Elegir los datos y formatos

Seleccione uno o varios niveles. Las cifras siguientes son valores de referencia de la última auditoría verificada y pueden cambiar si INETER actualiza la fuente:

- **Nivel 4** — 12 unidades; sin advertencias topológicas conocidas.
- **Nivel 5** — 68 unidades; 2 advertencias topológicas conocidas.
- **Nivel 6** — 491 unidades; 1 advertencia topológica conocida.
- **Nivel 7** — 2,337 unidades; 2 advertencias topológicas conocidas.

Una advertencia topológica no impide descargar ni conservar el KML oficial. Sin reparación, sólo se omiten los formatos analíticos de ese nivel; el proceso continúa con los demás. GeoPackage es el formato recomendado para análisis.

In [ ]:
if not globals().get("BOOTSTRAP_OK", False):
    raise RuntimeError(
        "La instalación no se completó. Ejecute primero la celda de instalación y corrija el error."
    )

import traceback
import uuid
from datetime import UTC, datetime
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display

from nica_geofetch.exceptions import NicaGeoFetchError
from nica_geofetch.logging_utils import configure_logging
from nica_geofetch.models import OutputFormat
from nica_geofetch.providers.ineter_pfafstetter import IneterPfafstetterProvider
from nica_geofetch.workflows import download_workflow, import_local_workflow

configure_logging()
provider = IneterPfafstetterProvider()
provider_selector = widgets.Dropdown(
    options=[("INETER Pfafstetter 2025", "ineter-pfafstetter")],
    description="Proveedor:",
)
level_checks = {
    4: widgets.Checkbox(
        value=True, description="Nivel 4 — 12 unidades; sin advertencias conocidas"
    ),
    5: widgets.Checkbox(value=False, description="Nivel 5 — 68 unidades; 2 advertencias conocidas"),
    6: widgets.Checkbox(value=False, description="Nivel 6 — 491 unidades; 1 advertencia conocida"),
    7: widgets.Checkbox(
        value=False, description="Nivel 7 — 2,337 unidades; 2 advertencias conocidas"
    ),
}
for control in level_checks.values():
    control.layout = widgets.Layout(width="440px")
select_all_button = widgets.Button(description="Seleccionar todos los niveles")
select_level4_button = widgets.Button(description="Seleccionar solo nivel 4")
format_selector = widgets.Dropdown(
    options=[
        ("KML original", "kml"),
        ("GeoPackage (recomendado)", "gpkg"),
        ("GeoJSON", "geojson"),
        ("Shapefile ZIP", "shapefile"),
        ("Todos", "all"),
    ],
    value="gpkg",
    description="Formato:",
)
repair_checkbox = widgets.Checkbox(
    value=False,
    description="Reparar geometrías inválidas para generar formatos analíticos",
    indent=False,
    layout=widgets.Layout(width="650px"),
)
repair_help = widgets.HTML(
    "<em>La reparación es opcional. El KML oficial original se conserva sin cambios. "
    "La reparación se aplica únicamente a una copia destinada a formatos analíticos "
    "y queda registrada en el manifiesto y la auditoría.</em>"
)
location_selector = widgets.RadioButtons(
    options=[
        ("Temporal de Colab", "temporary"),
        ("Google Drive (montar explícitamente)", "drive"),
    ],
    value="temporary",
    description="Destino:",
)
diagnose_button = widgets.Button(description="Diagnosticar acceso", button_style="info")
download_button = widgets.Button(description="Descargar y preparar", button_style="success")
zip_download_button = widgets.Button(
    description="Descargar ZIP a mi computadora",
    button_style="primary",
    disabled=True,
    layout=widgets.Layout(display="none", width="280px"),
)
progress = widgets.IntProgress(value=0, min=0, max=100, description="Progreso:")
status_label = widgets.HTML(value="Listo para comenzar.")
log_output = widgets.Output()
LAST_RESULT = None
LATEST_ARCHIVE = None
LEVEL_STATUS = {}


def select_all_levels(_button=None):
    for control in level_checks.values():
        control.value = True


def select_only_level4(_button=None):
    for level, control in level_checks.items():
        control.value = level == 4


def selected_levels():
    levels = [level for level, control in level_checks.items() if control.value]
    if not levels:
        raise ValueError("Seleccione por lo menos un nivel.")
    return levels


def selected_formats():
    if format_selector.value == "all":
        return list(OutputFormat)
    return [OutputFormat(format_selector.value)]


def new_output_directory(prefix):
    run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    if location_selector.value == "drive":
        from google.colab import drive

        drive.mount("/content/drive")
        base = Path("/content/drive/MyDrive/NicaGeoFetch_outputs")
    else:
        base = Path("/content/NicaGeoFetch_outputs")
    # Cada ejecución usa una carpeta nueva para no mezclar resultados anteriores.
    return base / f"{prefix}_{run_id}"


def spanish_summary(result):
    rows = []
    for row in result.summary_rows():
        rows.append(
            {
                "Nivel": row["level"],
                "KML oficial descargado": "Sí" if row["acquisition_valid"] else "No",
                "Estado de adquisición": "Correcto" if row["acquisition_valid"] else "Falló",
                "Estado geométrico": (
                    "Correcto"
                    if row["geometry_valid"]
                    else "Reparado"
                    if row["repair_applied"]
                    else "Con advertencias"
                ),
                "Geometrías con advertencias": row["invalid_geometry_count"],
                "Reparación solicitada": "Sí" if row["repair_requested"] else "No",
                "Reparación aplicada": "Sí" if row["repair_applied"] else "No",
                "Registros encontrados": row["features"],
                "Formatos analíticos generados": ", ".join(row["analytical_outputs"]) or "Omitido",
                "Advertencias": "; ".join(row["warnings"]) or "Ninguna",
                "Resultado": {
                    "correct": "Correcto",
                    "repaired": "Reparado",
                    "correct_with_warnings": "Correcto con advertencias",
                    "failed": "Falló",
                }[row["result"]],
            }
        )
    return pd.DataFrame(rows)


def show_progress(event, level, report):
    messages = {
        "generating_metadata": "Generando manifiesto, auditoría y checksums…",
        "creating_archive": "Creando ZIP final…",
        "completed": "Proceso completado.",
    }
    if event == "downloading" and level is not None:
        LEVEL_STATUS[level] = "Descargando"
        print(f"Descargando nivel {level}…")
    elif event == "validating" and level is not None:
        LEVEL_STATUS[level] = "Validando"
        print("Validando estructura del KML…")
        print("Revisando geometrías…")
    elif event == "source_preserved" and report is not None:
        if report.invalid_geometry_count:
            LEVEL_STATUS[report.level] = "Fuente conservada con advertencias"
            print(
                f"Nivel {report.level} descargado. Se detectaron "
                f"{report.invalid_geometry_count} advertencias topológicas."
            )
            print("El KML original se conservará.")
        else:
            LEVEL_STATUS[report.level] = "Fuente conservada"
            print(f"Nivel {report.level} descargado correctamente.")
    elif event == "analytical_conversion_completed" and report is not None:
        LEVEL_STATUS[report.level] = "Conversión analítica completada"
        print(f"Formatos analíticos del nivel {report.level} generados correctamente.")
    elif event == "analytical_skipped" and report is not None:
        LEVEL_STATUS[report.level] = "Completado con advertencias"
        requested = ", ".join(item.value for item in selected_formats() if item != OutputFormat.KML)
        if requested:
            print(
                f"No se generará {requested} para nivel {report.level} porque la reparación "
                "está desactivada o la fuente requiere revisión."
            )
        print("Continuando con el siguiente nivel…")
    elif event == "failed" and level is not None:
        LEVEL_STATUS[level] = "Falló"
    elif event in messages:
        print(messages[event])
    completed = sum(
        state
        in {
            "Fuente conservada",
            "Conversión analítica completada",
            "Completado con advertencias",
            "Falló",
        }
        for state in LEVEL_STATUS.values()
    )
    progress.value = min(95, int(95 * completed / max(1, len(LEVEL_STATUS))))
    status_label.value = "<b>Estado:</b> " + (
        messages.get(event) or (f"Nivel {level}: {LEVEL_STATUS.get(level, event)}")
    )


def diagnose_access(_button):
    diagnose_button.disabled = True
    try:
        with log_output:
            clear_output()
            level = selected_levels()[0]
            print(f"Conectando con el servicio oficial de INETER para diagnosticar nivel {level}…")
            report = provider.diagnose(level)
            if report.ok:
                print("El servicio respondió correctamente. Puede iniciar la descarga.")
            else:
                print(f"No se pudo confirmar el acceso: {report.message}")
                print("URL oficial:", report.official_url)
                print(
                    "Puede abrirla en un navegador, guardar el KML y usar la alternativa manual del paso 5."
                )
    except (NicaGeoFetchError, OSError, ValueError) as exc:
        with log_output:
            print(f"No se pudo completar el diagnóstico: {exc}")
    finally:
        diagnose_button.disabled = False


def download_and_validate(_button):
    global LAST_RESULT, LATEST_ARCHIVE, LEVEL_STATUS
    download_button.disabled = True
    diagnose_button.disabled = True
    zip_download_button.disabled = True
    zip_download_button.layout.display = "none"
    LAST_RESULT = None
    LATEST_ARCHIVE = None
    progress.value = 0
    try:
        levels = selected_levels()
        LEVEL_STATUS = {level: "En espera" for level in levels}
        formats = selected_formats()
        output = new_output_directory("descarga")
        with log_output:
            clear_output()
            print("Preparando la descarga…")
            print("Niveles seleccionados:", ", ".join(map(str, levels)) + ".")
            print("Formato solicitado:", ", ".join(item.value for item in formats) + ".")
            print(
                "Reparación geométrica:", "activada." if repair_checkbox.value else "desactivada."
            )
            print("Conectando con el servicio oficial de INETER…")
            # Las descargas son secuenciales para tratar al servicio institucional con cortesía.
            LAST_RESULT = download_workflow(
                levels=levels,
                formats=formats,
                output_directory=output,
                repair=repair_checkbox.value,
                progress_callback=show_progress,
            )
            progress.value = 100
            display(spanish_summary(LAST_RESULT))
            LATEST_ARCHIVE = LAST_RESULT.archive_path
            print("ZIP final:", LATEST_ARCHIVE)
            if location_selector.value == "drive":
                print(
                    "El archivo quedó guardado en Google Drive en la ruta exacta indicada arriba."
                )
            else:
                print(
                    "El archivo se creó temporalmente dentro de Colab. Use el botón "
                    "'Descargar ZIP a mi computadora' antes de cerrar la sesión."
                )
            zip_download_button.disabled = not LATEST_ARCHIVE.exists()
            zip_download_button.layout.display = (
                "inline-flex" if LATEST_ARCHIVE.exists() else "none"
            )
    except (NicaGeoFetchError, OSError, ValueError) as exc:
        with log_output:
            print(f"No se pudo completar el proceso: {exc}")
            print("No se conservó un archivo que no superó la validación de adquisición.")
            for level in locals().get("levels", []):
                print(f"URL oficial del nivel {level}: {provider.build_url(level)}")
            print(
                "Abra la URL en un navegador, guarde el KML y use la alternativa manual del paso 5."
            )
    except Exception:
        with log_output:
            print(
                "Ocurrió un error inesperado del programa. Puede revisar los detalles técnicos siguientes."
            )
            details = widgets.Accordion(
                children=[
                    widgets.Textarea(
                        value=traceback.format_exc(),
                        layout=widgets.Layout(width="100%", height="180px"),
                    )
                ]
            )
            details.set_title(0, "Detalles técnicos")
            display(details)
    finally:
        download_button.disabled = False
        diagnose_button.disabled = False


def download_latest_zip(_button):
    if LATEST_ARCHIVE is None or not Path(LATEST_ARCHIVE).is_file():
        return
    from google.colab import files

    # /content es temporal: la transferencia comienza únicamente tras este clic.
    files.download(str(LATEST_ARCHIVE))


select_all_button.on_click(select_all_levels)
select_level4_button.on_click(select_only_level4)
diagnose_button.on_click(diagnose_access)
download_button.on_click(download_and_validate)
zip_download_button.on_click(download_latest_zip)
display(provider_selector)
display(
    widgets.HBox(
        [
            widgets.VBox([level_checks[4], level_checks[5]]),
            widgets.VBox([level_checks[6], level_checks[7]]),
        ]
    )
)
display(widgets.HBox([select_all_button, select_level4_button]))
display(format_selector, repair_checkbox, repair_help, location_selector)
display(widgets.HTML("<h2>3. Descargar y preparar los archivos</h2>"))
display(widgets.HBox([diagnose_button, download_button]), progress, status_label, log_output)
display(zip_download_button)

## 4. Descargar el ZIP final

El botón azul **Descargar ZIP a mi computadora** aparece junto al flujo principal únicamente cuando el archivo más reciente existe. El ZIP incluye cada KML oficial conservado, los formatos analíticos que pudieron generarse, auditoría, manifiesto, procedencia y checksums.

Recuerde: `/content` pertenece a la sesión temporal de Colab. Los archivos no llegan automáticamente a su computadora y desaparecen cuando termina la sesión. El botón siguiente es un respaldo cercano; tampoco inicia la descarga hasta que usted hace clic.

In [ ]:
fallback_zip_button = widgets.Button(
    description="Descargar el último ZIP",
    button_style="primary",
    disabled=LATEST_ARCHIVE is None or not Path(LATEST_ARCHIVE).is_file(),
)


def download_fallback_zip(_button):
    if LATEST_ARCHIVE is None or not Path(LATEST_ARCHIVE).is_file():
        return
    from google.colab import files

    files.download(str(LATEST_ARCHIVE))


fallback_zip_button.on_click(download_fallback_zip)
display(fallback_zip_button)

## 5. Alternativa opcional: importar un KML descargado manualmente

Use esta sección únicamente si la descarga automática falla o si ya dispone de un KML oficial. **No necesita usarla después de una descarga automática correcta.** Permite validar, convertir y empaquetar sin descargar otra vez, o reprocesar exactamente el mismo archivo histórico.

Cuando falle el acceso automático, copie la URL oficial mostrada, ábrala en un navegador normal y guarde la respuesta como archivo `.kml`, sin evadir controles institucionales. Después elija aquí el nivel correcto y pulse **Cargar KML manualmente**.

In [ ]:
manual_level_selector = widgets.Dropdown(
    options=[("Nivel 4", 4), ("Nivel 5", 5), ("Nivel 6", 6), ("Nivel 7", 7)],
    value=4,
    description="Nivel:",
)
manual_repair_checkbox = widgets.Checkbox(
    value=False,
    description="Reparar geometrías inválidas para formatos analíticos",
    indent=False,
    layout=widgets.Layout(width="620px"),
)
manual_upload_button = widgets.Button(
    description="Cargar KML manualmente",
    button_style="warning",
)
manual_zip_button = widgets.Button(
    description="Descargar ZIP importado",
    button_style="primary",
    disabled=True,
    layout=widgets.Layout(display="none", width="240px"),
)
manual_status_output = widgets.Output()
MANUAL_LAST_RESULT = None
MANUAL_LATEST_ARCHIVE = None


def upload_manual_kml(_button):
    global LAST_RESULT, LATEST_ARCHIVE, MANUAL_LAST_RESULT, MANUAL_LATEST_ARCHIVE
    manual_upload_button.disabled = True
    manual_zip_button.disabled = True
    manual_zip_button.layout.display = "none"
    MANUAL_LAST_RESULT = None
    MANUAL_LATEST_ARCHIVE = None
    try:
        # La carga manual es opcional y el selector sólo se abre tras este clic.
        from google.colab import files

        with manual_status_output:
            clear_output()
            print("Seleccione un único archivo KML oficial guardado previamente.")
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith(".kml")]
        if len(candidates) != 1:
            raise ValueError("Cargue exactamente un archivo .kml.")
        manual_path = Path(candidates[0])
        manual_output = new_output_directory("importacion_manual")
        with manual_status_output:
            print("Validando estructura del KML y revisando geometrías…")
            MANUAL_LAST_RESULT = import_local_workflow(
                input_path=manual_path,
                level=manual_level_selector.value,
                formats=selected_formats(),
                output_directory=manual_output,
                repair=manual_repair_checkbox.value,
            )
            LAST_RESULT = MANUAL_LAST_RESULT
            LATEST_ARCHIVE = MANUAL_LAST_RESULT.archive_path
            MANUAL_LATEST_ARCHIVE = MANUAL_LAST_RESULT.archive_path
            display(spanish_summary(MANUAL_LAST_RESULT))
            print("ZIP final:", MANUAL_LATEST_ARCHIVE)
            print("Descárguelo antes de cerrar Colab si está dentro de /content.")
        manual_zip_button.disabled = not MANUAL_LATEST_ARCHIVE.exists()
        manual_zip_button.layout.display = (
            "inline-flex" if MANUAL_LATEST_ARCHIVE.exists() else "none"
        )
    except (NicaGeoFetchError, OSError, ValueError) as exc:
        with manual_status_output:
            print(f"No se pudo importar el KML del nivel {manual_level_selector.value}: {exc}")
            print(
                "Revise que eligió el nivel correcto y que el archivo es un KML vectorial oficial."
            )
    except Exception:
        with manual_status_output:
            print("Ocurrió un error inesperado durante la importación manual.")
            details = widgets.Accordion(
                children=[
                    widgets.Textarea(
                        value=traceback.format_exc(),
                        layout=widgets.Layout(width="100%", height="180px"),
                    )
                ]
            )
            details.set_title(0, "Detalles técnicos")
            display(details)
    finally:
        manual_upload_button.disabled = False


def download_manual_zip(_button):
    if MANUAL_LATEST_ARCHIVE is None or not Path(MANUAL_LATEST_ARCHIVE).is_file():
        return
    from google.colab import files

    files.download(str(MANUAL_LATEST_ARCHIVE))


manual_upload_button.on_click(upload_manual_kml)
manual_zip_button.on_click(download_manual_zip)
display(manual_level_selector, manual_repair_checkbox, manual_upload_button)
display(manual_status_output, manual_zip_button)

## Cómo interpretar el resultado

**Correcto con advertencias** significa que el KML oficial fue adquirido y conservado, pero alguna geometría requiere reparación explícita antes de crear formatos analíticos. No significa que el archivo KML sea inutilizable. **Reparado** significa que el original quedó intacto y sólo la copia de trabajo analítica fue reparada y registrada.

In [ ]:
if LAST_RESULT is None:
    print("Use el botón del paso 3 o la alternativa manual del paso 5 para crear un resultado.")
else:
    display(spanish_summary(LAST_RESULT))
    print("Último ZIP:", LAST_RESULT.archive_path)